In [5]:
import numpy as np
from pathlib import Path
import pandas as pd
try:
    from Preprocessing.analysis_utils import get_stimulus_timestamps, get_running_timestamps, get_running_df, resample_dff_for_trial, resample_running_for_trial
except ModuleNotFoundError:
    from analysis_utils import get_stimulus_timestamps, get_running_timestamps, get_running_df, resample_dff_for_trial, resample_running_for_trial
from hdmf_zarr.nwb import NWBZarrIO
import matplotlib.pyplot as plt
import anndata as ad
import pickle
from scipy.interpolate import interp1d
from pathlib import Path
import os
import time
import zarr
import numcodecs
import shutil
RESAMPLE_RATE = 10    # Hz
PRE_STIM      = 1.0   # seconds before stimulus onset
POST_STIM     = 1.0   # seconds after stimulus offset

SCRATCH_DIR = Path("/root/capsule/scratch")
DATA_DIR = Path("/root/capsule/data")
OPHYS_DIR = SCRATCH_DIR / "ophys"
OPHYS_DIR.mkdir(exist_ok=True)

STIM_ALIGNED_DIR = OPHYS_DIR / "stim_aligned"
STIM_ALIGNED_DIR.mkdir(exist_ok=True)

load data information

In [3]:
data_info = pickle.load(open('/root/capsule/code/Preprocessing/data_info.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

mouse_ids: [778174, 786297, 797371]


# Generate Stimulus-Aligned DFF Traces (Resampled at 10 Hz)

For each mouse × session × plane × stimulus trial, extract dff traces resampled at 10 Hz in a window from **1 s before stimulus onset** to **1 s after stimulus offset**.


In [6]:
# ── Main processing loop ────────────────────────────────────────────
# Process all mice × STAGE_1 sessions × planes × stimuli

# Output structure:
#   session_result[stim_name] = {
#       'running':       np.ndarray (trials, time, 2)  [speed, speed_filtered] float32
#       'time_relative': np.ndarray (time,)            seconds relative to stim onset
#       'trial_info':    pd.DataFrame          stimulus attributes per trial
#       'dff': {
#           plane: {'data': np.ndarray (trials, time, cells), 'cell_ids': np.ndarray (cells,)}
#       }
#   }

for mouse_id in mouse_ids:
    stage1 = {
        s: info for s, info in data_info[mouse_id]['ophys'].items()
        if info['session_type'] == 'STAGE_1'
    }

    for session_name, session_info in stage1.items():
        out_path = STIM_ALIGNED_DIR / f"{mouse_id}_{session_name}.pkl"
        if out_path.exists():
            print(f"[skip] {out_path.name} already exists")
            continue

        print(f"\n{'='*60}")
        print(f"Mouse {mouse_id}  ·  {session_name}  ·  {session_info['raw']}")
        print(f"{'='*60}")

        # ── 1. Load stimulus table & running speed ──────────────────
        ophys_raw_path = DATA_DIR / session_info['raw']

        sync_file_path = list((ophys_raw_path / 'behavior').glob("*.h5"))[0]
        stim_pkl_file_path = list((ophys_raw_path / 'behavior').glob("*.pkl"))[0]
        running_timestamps = get_running_timestamps(sync_file_path)
        running_speed_df = get_running_df(stim_pkl_file_path, running_timestamps)

        stim_table_path = list((ophys_raw_path / 'behavior').glob("*stim_table.csv"))[0]
        stimulus_table = pd.read_csv(stim_table_path)
        stim_names = stimulus_table['stim_name'].unique()
        print(f"  Stimulus table: {len(stimulus_table)} trials, types = {list(stim_names)}")

        # ── 2. Open NWB (all planes) ───────────────────────────────
        ophys_processed_path = DATA_DIR / session_info['processed']
        nwb_zarr_path = list(ophys_processed_path.glob("*.nwb"))[0]

        # Pre-build per-stim containers with stimulus-aligned running
        session_result = {}
        for stim_name in stim_names:
            stim_group = stimulus_table[stimulus_table['stim_name'] == stim_name].copy()
            stim_group = stim_group.reset_index(drop=True)
            n_trials = len(stim_group)

            # Resample running speed for each trial
            trials_running = []
            trials_time    = []
            for _, trial in stim_group.iterrows():
                run_r, t_r = resample_running_for_trial(
                    running_speed_df,
                    trial['start_time'], trial['stop_time'],
                )
                trials_running.append(run_r)
                trials_time.append(t_r)

            max_len      = max(t.shape[0] for t in trials_time)
            ref_time_idx = int(np.argmax([t.shape[0] for t in trials_time]))
            time_common  = trials_time[ref_time_idx]

            running_stacked = np.full(
                (n_trials, max_len, 2), np.nan, dtype=np.float32
            )
            for i, run_trial in enumerate(trials_running):
                running_stacked[i, :run_trial.shape[0], :] = run_trial

            session_result[stim_name] = {
                'running':       running_stacked,   # (n_trials, n_timepoints, 2) [speed, speed_filtered]
                'time_relative': time_common,
                'trial_info':    stim_group,
                'dff': {},
            }

        # ── 2b. Loop over planes and extract dff ───────────────────
        with NWBZarrIO(path=nwb_zarr_path, mode='r') as io:
            nwbfile = io.read()
        # with NWBHDF5IO(str(nwb_zarr_path), mode='r', driver='zarr') as io:
            # nwbfile = io.read()
            planes = list(nwbfile.processing.keys())

            for plane in planes:
                print(f"  Plane {plane} ... ", end="", flush=True)
                proc = nwbfile.processing[plane]['dff_timeseries']['dff_timeseries']
                dff_data   = proc.data[:]          # (n_timepoints, n_cells)
                dff_ts     = proc.timestamps[:]    # (n_timepoints,)
                cell_ids   = np.arange(1,dff_data.shape[1]+1)
                n_cells    = dff_data.shape[1]

                for stim_name in stim_names:
                    stim_group   = session_result[stim_name]['trial_info']
                    time_common  = session_result[stim_name]['time_relative']
                    n_trials     = len(stim_group)
                    max_len      = time_common.shape[0]

                    trials_dff  = []
                    trials_time = []
                    for _, trial in stim_group.iterrows():
                        dff_r, t_r = resample_dff_for_trial(
                            dff_data, dff_ts,
                            trial['start_time'], trial['stop_time'],
                        )
                        trials_dff.append(dff_r)
                        trials_time.append(t_r)

                    # Pad to longest time grid
                    dff_stacked = np.full(
                        (n_trials, max_len, n_cells), np.nan, dtype=np.float32
                    )
                    for i, dff_trial in enumerate(trials_dff):
                        dff_stacked[i, :dff_trial.shape[0], :] = dff_trial

                    # Store per-plane dff
                    session_result[stim_name]['dff'][plane] = {
                        'data':     dff_stacked,
                        'cell_ids': cell_ids,
                    }

                shapes_str = ", ".join(
                    f"{sn}: {session_result[sn]['dff'][plane]['data'].shape}"
                    for sn in stim_names
                )
                print(f"done  [{shapes_str}]")

        # ── 3. Save per-session pickle ──────────────────────────────
        with open(out_path, 'wb') as f:
            pickle.dump(session_result, f, protocol=pickle.HIGHEST_PROTOCOL)
        size_mb = out_path.stat().st_size / 1e6

print("\n✅ All sessions processed.")


Mouse 778174  ·  session_1  ·  multiplane-ophys_778174_2025-03-14_11-17-01


/opt/conda/envs/ophys_env/lib/python3.12/site-packages/pandas/compat/pickle_compat.py:35: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)


  Stimulus table: 1124 trials, types = ['spontaneous', 'drifting_gratings_contrast', 'drifting_gratings_TF']
  Plane VISp_0 ... done  [spontaneous: (4, 3634, 162), drifting_gratings_contrast: (560, 41, 162), drifting_gratings_TF: (560, 41, 162)]
  Plane VISp_1 ... done  [spontaneous: (4, 3634, 253), drifting_gratings_contrast: (560, 41, 253), drifting_gratings_TF: (560, 41, 253)]
  Plane VISp_2 ... done  [spontaneous: (4, 3634, 432), drifting_gratings_contrast: (560, 41, 432), drifting_gratings_TF: (560, 41, 432)]
  Plane VISp_3 ... done  [spontaneous: (4, 3634, 300), drifting_gratings_contrast: (560, 41, 300), drifting_gratings_TF: (560, 41, 300)]
  Plane VISp_4 ... done  [spontaneous: (4, 3634, 428), drifting_gratings_contrast: (560, 41, 428), drifting_gratings_TF: (560, 41, 428)]
  Plane VISp_5 ... done  [spontaneous: (4, 3634, 344), drifting_gratings_contrast: (560, 41, 344), drifting_gratings_TF: (560, 41, 344)]
  Plane VISp_6 ... done  [spontaneous: (4, 3634, 443), drifting_grati

/opt/conda/envs/ophys_env/lib/python3.12/site-packages/pandas/compat/pickle_compat.py:35: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)


  Stimulus table: 1124 trials, types = ['spontaneous', 'drifting_gratings_contrast', 'drifting_gratings_TF']
  Plane VISp_0 ... done  [spontaneous: (4, 3634, 161), drifting_gratings_contrast: (560, 41, 161), drifting_gratings_TF: (560, 41, 161)]
  Plane VISp_1 ... done  [spontaneous: (4, 3634, 245), drifting_gratings_contrast: (560, 41, 245), drifting_gratings_TF: (560, 41, 245)]
  Plane VISp_2 ... done  [spontaneous: (4, 3634, 438), drifting_gratings_contrast: (560, 41, 438), drifting_gratings_TF: (560, 41, 438)]
  Plane VISp_3 ... done  [spontaneous: (4, 3634, 307), drifting_gratings_contrast: (560, 41, 307), drifting_gratings_TF: (560, 41, 307)]
  Plane VISp_4 ... done  [spontaneous: (4, 3634, 438), drifting_gratings_contrast: (560, 41, 438), drifting_gratings_TF: (560, 41, 438)]
  Plane VISp_5 ... done  [spontaneous: (4, 3634, 356), drifting_gratings_contrast: (560, 41, 356), drifting_gratings_TF: (560, 41, 356)]
  Plane VISp_6 ... done  [spontaneous: (4, 3634, 449), drifting_grati

/opt/conda/envs/ophys_env/lib/python3.12/site-packages/pandas/compat/pickle_compat.py:35: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)


  Stimulus table: 1124 trials, types = ['spontaneous', 'drifting_gratings_contrast', 'drifting_gratings_TF']
  Plane VISp_0 ... done  [spontaneous: (4, 3634, 149), drifting_gratings_contrast: (560, 41, 149), drifting_gratings_TF: (560, 41, 149)]
  Plane VISp_1 ... done  [spontaneous: (4, 3634, 219), drifting_gratings_contrast: (560, 41, 219), drifting_gratings_TF: (560, 41, 219)]
  Plane VISp_2 ... 

KeyboardInterrupt: 